In [ ]:
import pandas as pd

# ============================================================
# CẤU HÌNH 
# ============================================================
MAIN_FILE = "D:\\UIT\\KLTN\\data\\final\\vismishds_phase1_final.csv"
SUB_FILE  = "D:\\UIT\\KLTN\\data\\hard_negative\\hard_negative_final.csv"
OUTPUT_FILE = "D:\\UIT\\KLTN\\model\\base\\merged_dataset.csv"
RANDOM_SEED = 42  
# ============================================================

def main():
    # 1. Đọc dữ liệu
    print("📂 Đọc file...")
    df_main = pd.read_csv(MAIN_FILE)
    df_sub  = pd.read_csv(SUB_FILE)

    print(f"   File chính : {len(df_main):,} rows")
    print(f"   File phụ   : {len(df_sub):,} rows")

    # 2. Lọc nhóm ứng viên: label=1 VÀ data_origin=synthetic
    mask_candidates = (df_main["label"] == 1) & (df_main["data_origin"] == "synthetic")
    df_candidates   = df_main[mask_candidates]
    df_rest         = df_main[~mask_candidates]

    n_sub        = len(df_sub)
    n_candidates = len(df_candidates)

    print(f"\n🔍 Nhóm ứng viên (label=1 & data_origin=synthetic): {n_candidates:,} rows")
    print(f"   Số row cần thay thế (= số row file phụ)        : {n_sub:,} rows")

    # Kiểm tra điều kiện
    if n_sub > n_candidates:
        raise ValueError(
            f"❌ File phụ có {n_sub} rows nhưng nhóm ứng viên chỉ có {n_candidates} rows. "
            "Không đủ để thay thế!"
        )

    # 3. Random sample N row từ nhóm ứng viên để xóa
    df_sampled_out = df_candidates.sample(n=n_sub, random_state=RANDOM_SEED)
    df_kept        = df_candidates.drop(df_sampled_out.index)  # phần ứng viên còn lại (không bị thay)

    print(f"\n✂️  Đã random chọn {n_sub:,} rows từ nhóm ứng viên để xóa")
    print(f"   Ứng viên còn lại (giữ nguyên): {len(df_kept):,} rows")

    # 4. Ghép lại: phần không phải ứng viên + ứng viên còn lại + file phụ
    df_output = pd.concat([df_rest, df_kept, df_sub], ignore_index=True)

    # 5. Shuffle toàn bộ dataset
    df_output = df_output.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)

    print(f"\n🔀 Đã shuffle toàn bộ dataset")
    print(f"   Tổng rows output: {len(df_output):,} rows")

    # 6. Xuất file
    df_output.to_csv(OUTPUT_FILE, index=False)
    print(f"\n✅ Đã lưu file output: {OUTPUT_FILE}")

    # Thống kê nhanh
    print("\n📊 Thống kê label trong output:")
    print(df_output["label"].value_counts().to_string())

    if "data_origin" in df_output.columns:
        print("\n📊 Thống kê data_origin trong output:")
        print(df_output["data_origin"].value_counts().to_string())


if __name__ == "__main__":
    main()

📂 Đọc file...
   File chính : 10,562 rows
   File phụ   : 653 rows

🔍 Nhóm ứng viên (label=1 & data_origin=synthetic): 4,996 rows
   Số row cần thay thế (= số row file phụ)        : 653 rows

✂️  Đã random chọn 653 rows từ nhóm ứng viên để xóa
   Ứng viên còn lại (giữ nguyên): 4,343 rows

🔀 Đã shuffle toàn bộ dataset
   Tổng rows output: 10,562 rows

✅ Đã lưu file output: D:\UIT\KLTN\model\base\temp.csv

📊 Thống kê label trong output:
label
0    5320
1    5242

📊 Thống kê data_origin trong output:
data_origin
synthetic                  7342
real                       2567
synthetic_hard_negative     653


In [ ]:
# ============================================================
# CẤU HÌNH 
# ============================================================
MAIN_FILE   = "D:\\UIT\\KLTN\\model\\base\\merged_dataset.csv"   
SUB_FILE_A  = "D:\\UIT\\KLTN\\data\\external\\vilexnorm\\curated\\vilexnorm_clean_p2p_label0.csv"   
SUB_FILE_B  = "D:\\UIT\\KLTN\\data\\external\\vilexnorm\\curated\\vilexnorm_hard_negative_label0.csv"  
OUTPUT_FILE = "D:\\UIT\\KLTN\\model\\base\\merged_dataset.csv"
RANDOM_SEED = 42
# ============================================================

def align_columns(df_sub: pd.DataFrame, columns: list) -> pd.DataFrame:
    """
    Chỉ giữ các cột khớp với file chính, thêm cột còn thiếu với giá trị NaN.
    Không thêm cột lạ từ file phụ vào.
    """
    df_aligned = pd.DataFrame(columns=columns)
    for col in columns:
        if col in df_sub.columns:
            df_aligned[col] = df_sub[col].values
        else:
            df_aligned[col] = None
    return df_aligned


def replace_batch(df_main: pd.DataFrame, df_sub: pd.DataFrame,
                  sub_name: str, seed_offset: int) -> pd.DataFrame:
    """
    Thay một batch ứng viên (label=0 & data_origin=synthetic) bằng df_sub.
    """
    mask_candidates = (df_main["label"] == 0) & (df_main["data_origin"] == "synthetic")
    df_candidates   = df_main[mask_candidates]
    df_rest         = df_main[~mask_candidates]

    n_sub        = len(df_sub)
    n_candidates = len(df_candidates)

    print(f"\n📂 Xử lý {sub_name}: {n_sub:,} rows")
    print(f"   Ứng viên còn lại (label=0 & data_origin=synthetic): {n_candidates:,} rows")

    if n_sub > n_candidates:
        raise ValueError(
            f"❌ {sub_name} có {n_sub} rows nhưng nhóm ứng viên chỉ còn {n_candidates} rows. "
            "Không đủ để thay thế!"
        )

    # Random sample N row ứng viên để xóa
    df_sampled_out = df_candidates.sample(n=n_sub, random_state=RANDOM_SEED + seed_offset)
    df_kept        = df_candidates.drop(df_sampled_out.index)

    print(f"   ✂️  Đã xóa {n_sub:,} rows ứng viên")
    print(f"   Ứng viên còn lại sau khi xóa: {len(df_kept):,} rows")

    # Align cột file phụ theo cột file chính
    df_sub_aligned = align_columns(df_sub, df_main.columns.tolist())

    # Ghép lại
    df_result = pd.concat([df_rest, df_kept, df_sub_aligned], ignore_index=True)
    return df_result


def main():
    # 1. Đọc dữ liệu
    print("📂 Đọc file...")
    df_main  = pd.read_csv(MAIN_FILE)
    df_sub_a = pd.read_csv(SUB_FILE_A)
    df_sub_b = pd.read_csv(SUB_FILE_B)

    print(f"   File chính  : {len(df_main):,} rows | cột: {df_main.columns.tolist()}")
    print(f"   File phụ A  : {len(df_sub_a):,} rows | cột: {df_sub_a.columns.tolist()}")
    print(f"   File phụ B  : {len(df_sub_b):,} rows | cột: {df_sub_b.columns.tolist()}")

    # 2. Xử lý file phụ A
    df_after_a = replace_batch(df_main, df_sub_a, sub_name="File phụ A", seed_offset=0)

    # 3. Xử lý file phụ B (dùng output sau khi đã xử lý A)
    df_after_b = replace_batch(df_after_a, df_sub_b, sub_name="File phụ B", seed_offset=1)

    # 4. Shuffle toàn bộ
    df_output = df_after_b.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)
    print(f"\n🔀 Đã shuffle toàn bộ dataset")
    print(f"   Tổng rows output: {len(df_output):,} rows")

    # 5. Xuất file (giữ nguyên cột file chính)
    df_output.to_csv(OUTPUT_FILE, index=False)
    print(f"\n✅ Đã lưu file output: {OUTPUT_FILE}")

    # Thống kê nhanh
    print("\n📊 Thống kê label trong output:")
    print(df_output["label"].value_counts().to_string())

    if "data_origin" in df_output.columns:
        print("\n📊 Thống kê data_origin trong output:")
        print(df_output["data_origin"].value_counts().to_string())


if __name__ == "__main__":
    main()

📂 Đọc file...
   File chính  : 10,562 rows | cột: ['sample_id', 'content', 'label', 'has_url', 'has_phone_number', 'sender_type', 'category', 'obfuscation_level', 'data_origin', 'source_dataset', 'source_file', 'source_row_id']
   File phụ A  : 500 rows | cột: ['source_dataset', 'source_file', 'source_row_id', 'split', 'original', 'normalized', 'content', 'label', 'has_url', 'has_phone_number', 'sender_type', 'category', 'obfuscation_level', 'data_origin', 'external_source_type', 'text_variant_type', 'candidate_type', 'hard_case_type', 'review_status', 'filter_reason', 'reject_reason', 'flags']
   File phụ B  : 501 rows | cột: ['sample_id', 'content', 'label', 'has_url', 'has_phone_number', 'sender_type', 'category', 'obfuscation_level', 'data_origin', 'source_dataset', 'source_file', 'source_row_id', 'external_source_type', 'text_variant_type', 'hard_case_type', 'review_status', 'original', 'normalized']

📂 Xử lý File phụ A: 500 rows
   Ứng viên còn lại (label=0 & data_origin=syntheti

In [9]:
import pandas as pd
from pathlib import Path

# ─────────────────────────────────────────────────────────────────────────────
# CẤU HÌNH 
# ─────────────────────────────────────────────────────────────────────────────
ORIGINAL_FILE  = "D:\\UIT\\KLTN\\model\\base\\merged_dataset.csv"          
REPLACE_FILE   = "D:\\UIT\\KLTN\\cleaned_paraphrased_dataset_094253.csv"
OUTPUT_FILE    = "D:\\UIT\\KLTN\\model\\base\\temp.csv" 

ID_COL         = "sample_id"             
COLUMNS_TO_REPLACE = None                # None = thay tất cả cột (trừ ID_COL)
                                         # hoặc chỉ định: ["content", "label"]
# ─────────────────────────────────────────────────────────────────────────────


def load_file(path: str) -> pd.DataFrame:
    """Load CSV hoặc Excel tự động theo đuôi file."""
    p = Path(path)
    if p.suffix.lower() in (".xlsx", ".xls"):
        return pd.read_excel(path)
    return pd.read_csv(path)


def replace_by_sample_id(
    original_file: str,
    replace_file: str,
    output_file: str,
    id_col: str,
    columns_to_replace: list | None = None,
) -> pd.DataFrame:

    df_orig    = load_file(original_file)
    df_replace = load_file(replace_file)

    # Xác định cột sẽ được thay thế
    if columns_to_replace is None:
        columns_to_replace = [c for c in df_replace.columns if c != id_col]

    print(f"File gốc       : {original_file}  →  {len(df_orig)} dòng")
    print(f"File thay thế  : {replace_file}   →  {len(df_replace)} dòng")
    print(f"Cột key        : {id_col}")
    print(f"Cột thay thế   : {columns_to_replace}\n")

    # Đặt sample_id làm index để update nhanh
    df_orig    = df_orig.set_index(id_col)
    df_replace = df_replace.set_index(id_col)[columns_to_replace]

    # Log trước khi thay
    matched_ids = df_replace.index.intersection(df_orig.index)
    print(f"Số dòng sẽ được cập nhật: {len(matched_ids)}")

    # Thực hiện thay thế (chỉ các cột chỉ định)
    df_orig.update(df_replace)

    df_result = df_orig.reset_index()

    # Lưu file
    out = Path(output_file)
    if out.suffix.lower() in (".xlsx", ".xls"):
        df_result.to_excel(output_file, index=False)
    else:
        df_result.to_csv(output_file, index=False)

    print(f"\n✅ Đã lưu kết quả vào: {output_file}  ({len(df_result)} dòng)")
    return df_result


# ─────────────────────────────────────────────────────────────────────────────
# Chạy
# ─────────────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    df_updated = replace_by_sample_id(
        original_file      = ORIGINAL_FILE,
        replace_file       = REPLACE_FILE,
        output_file        = OUTPUT_FILE,
        id_col             = ID_COL,
        columns_to_replace = COLUMNS_TO_REPLACE,
    )

    # Preview 5 dòng đầu kết quả
    print("\nPreview kết quả:")
    print(df_updated.head())

File gốc       : D:\UIT\KLTN\model\base\merged_dataset.csv  →  10562 dòng
File thay thế  : D:\UIT\KLTN\cleaned_paraphrased_dataset_094253.csv   →  4333 dòng
Cột key        : sample_id
Cột thay thế   : ['content', 'label', 'has_url', 'has_phone_number', 'sender_type', 'category', 'obfuscation_level', 'data_origin', 'source_dataset', 'source_file', 'source_row_id']

Số dòng sẽ được cập nhật: 4333


C:\Users\Nam Hai\AppData\Local\Temp\ipykernel_8392\1754866632.py:54: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[1453 656 145 ... 'phase1_08888' 144 'phase1_08732']' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df_orig.update(df_replace)



✅ Đã lưu kết quả vào: D:\UIT\KLTN\model\base\temp.csv  (10562 dòng)

Preview kết quả:
       sample_id                                            content  label  \
0   phase1_01451  (TB) SINH NHẬT VINA - NẠP THẺ CÓ QUÀ! VinaPhon...      0   
1   phase1_00657  Quy khach duoc cong them 8 diem Viettel++ boi ...      0   
2  hardneg_00205  Cuc Thue: Nguyen Thi Xuan can bo sung giay to ...      1   
3   phase1_10459  Dịch vụ Quản lý Danh tính Quốc gia: Tài khoản ...      1   
4   phase1_08523  [VPBank] Tài khoản ngưng hoạt động đột ngột. T...      1   

   has_url  has_phone_number      sender_type           category  \
0        1                 1        brandname         Viễn thông   
1        1                 0        brandname               Khác   
2        0                 0        brandname   Dịch vụ công giả   
3        1                 0  personal_number   Dịch vụ công giả   
4        1                 0        brandname  Giả mạo ngân hàng   

                      obfuscation_l

# RECONSTRUCT DATASET

In [ ]:
import pandas as pd
import numpy as np
import re
import math
import argparse
import os
from collections import defaultdict
from underthesea import word_tokenize
df=pd.read_csv("D:\\UIT\\KLTN\\model\\base\\temp.csv")

1. Tự động đánh số sample_id

In [9]:

df["sample_id"]=[f"ViSmish_{i:05d}" for i in range(len(df))]
df.to_csv("D:\\UIT\\KLTN\\model\\base\\temp.csv", index=False)

2. Tự động phân loại obfuscation_level theo ViSmishDS_Data_Augmentation.md

In [1]:
"""
classify_obfuscation.py
------------------------
Phân loại cột obfuscation_level cho dataset ViSmishDS.

Logic:
  - label == 0  →  obfuscation_level = 'NONE'
  - label == 1  →  phân tích cột 'content', gán một trong 6 level:
      FORMAL        (level 0) – không có obfuscation
      LEET_LIGHT    (level 1) – thay 1-2 vowel bằng số: o→0, a→4, i→1, e→3
      LEET_HEAVY    (level 2) – nhiều ký tự bị thay, ! thay chữ, random code
      DOT_DASH      (level 3) – chèn - hoặc . giữa ký tự (escalate từ 1,2)
      MIXED_SPECIAL (level 4) – hỗn hợp nhiều ký tự đặc biệt lẫn vào text
      EXTREME_NOISE (level 5) – Unicode diacritics bị biến dạng, gần như không đọc được

Cách dùng:
    python classify_obfuscation.py --input dataset.csv --output dataset_labeled.csv
"""

import re
import argparse
import pandas as pd


# ──────────────────────────────────────────────
# Hàm detect từng level (escalating: cao nhất khớp → chọn level đó)
# ──────────────────────────────────────────────

def has_extreme_noise(text: str) -> bool:
    """
    Level 5: Unicode diacritics tiếng Việt bị biến dạng nặng.
    Dấu hiệu: tổ hợp ký tự Unicode bất thường xuất hiện dày đặc,
    hoặc các ký tự như ỢờỘỤ, ắ, ỉ, ễ... xen lẫn Latin và ký tự đặc biệt.
    """
    # Đếm số ký tự thuộc khối Vietnamese Unicode (U+1EA0–U+1EF9)
    viet_unicode = re.findall(r'[\u1ea0-\u1ef9]', text)
    # Đếm ký tự đặc biệt (không phải chữ, số, khoảng trắng, dấu câu thông thường)
    special_chars = re.findall(r'[^\w\s\.,;:\-\!\?\(\)\/\+\%\'\"@#\u00c0-\u024f]', text)

    total_len = max(len(text), 1)
    viet_ratio = len(viet_unicode) / total_len
    special_ratio = len(special_chars) / total_len

    # Extreme noise: tiếng Việt bị biến dạng kết hợp nhiều ký tự lạ
    return viet_ratio > 0.08 and special_ratio > 0.05


RE_INLINE_SPECIALS = re.compile(
    r'(?<=[a-zA-Z0-9\u00C0-\u1EF9])'             # Đứng ngay sau chữ/số
    r'[";\*\^\$~`\|\\+=\{\}\[\]\<\>\#\%]'        # Tập ký tự nhiễu (KHÔNG chứa . - _ !)
    r'|'                                         # HOẶC
    r'[";\*\^\$~`\|\\+=\{\}\[\]\<\>\#\%]'        # Đứng ngay trước chữ/số
    r'(?=[a-zA-Z0-9\u00C0-\u1EF9])'
)

RE_ALL_SPECIALS = re.compile(r'[";\*\^\$~`\|\\+=\{\}\[\]\<\>\#\%\!\@\&\?:]')

def has_mixed_special(text: str) -> bool:
    """
    Level 4: Hỗn hợp nhiều ký tự đặc biệt lẫn vào giữa text.
    Cải tiến: Bắt đa dạng ký tự hơn và đo lường độ đa dạng của nhiễu.
    """
    inline_specials = RE_INLINE_SPECIALS.findall(text)
    all_specials = RE_ALL_SPECIALS.findall(text)
    
    total_len = max(len(text), 1)
    
    # Nếu một tin nhắn chứa >= 4 loại ký tự đặc biệt khác nhau -> Chắc chắn là hỗn hợp (Level 4)
    unique_specials = set(all_specials)
    
    return len(inline_specials) >= 3 or (len(all_specials) / total_len > 0.08) or len(unique_specials) >= 4


def has_dot_dash(text: str) -> bool:
    """
    Level 3: Chèn dấu - hoặc . giữa các ký tự (tách từng chữ cái).
    Dấu hiệu: pattern X-X-X hoặc X.X.X với X là chữ/số đơn lẻ.
    """
    # Pattern: ít nhất 3 ký tự đơn liên tiếp được nối bằng - hoặc .
    dot_dash_pattern = re.compile(r'[a-zA-Z0-9\u00c0-\u024f][-\.][a-zA-Z0-9\u00c0-\u024f][-\.][a-zA-Z0-9\u00c0-\u024f]')
    return bool(dot_dash_pattern.search(text))


def has_leet_heavy(text: str) -> bool:
    """
    Level 2: Leet nặng — nhiều ký tự bị thay thế, ! thay chữ, random code cuối.
    Dấu hiệu:
      - ! thay chữ cái (d!eu, k!en, t!en)
      - Nhiều leet cùng lúc (>= 3 lần thay thế)
      - Random alphanumeric code cuối tin (pattern: khoảng trắng + 4-8 ký tự ngẫu nhiên)
    """
    # ! thay thế chữ cái trong từ
    excl_replace = re.findall(r'[a-zA-Z]![a-zA-Z]|[a-zA-Z]{1,3}![a-zA-Z]{1,3}', text)

    # Đếm tổng số leet substitution
    leet_subs = re.findall(r'(?<=[a-zA-Z])[014@][a-zA-Z]|[a-zA-Z][014@](?=[a-zA-Z])', text)

    # Random code cuối: chuỗi alphanumeric 4-8 ký tự viết hoa/số ngẫu nhiên
    random_code = re.findall(r'\b[A-Z0-9]{4,8}\b', text)

    return len(excl_replace) >= 1 or len(leet_subs) >= 3 or len(random_code) >= 2


def has_leet_light(text: str) -> bool:
    """
    Level 1: Leet nhẹ — thay 1-2 vowel bằng số.
    Dấu hiệu: o→0, a→4, i→1, e→3 xuất hiện trong từ.
    """
    # Tìm số xuất hiện trong/giữa chuỗi chữ cái (dấu hiệu leet)
    leet_in_word = re.findall(
        r'[a-zA-Z][014][a-zA-Z]'     # số nằm giữa chữ: t0ng, b4n, d1nh
        r'|[a-zA-Z]{2,}[014]'        # số ở cuối từ: ng0, qu4
        r'|[014][a-zA-Z]{2,}',       # số ở đầu từ: 0ng, 4nh
        text
    )
    return len(leet_in_word) >= 1


def classify_obfuscation(text: str) -> str:
    """
    Phân loại obfuscation level cho label=1.
    Escalating: kiểm tra từ cao nhất (level 5) xuống thấp nhất (level 0).
    """
    if not isinstance(text, str) or text.strip() == '':
        return 'Level 0 - Formal'

    if has_extreme_noise(text):
        return 'Level 5 - Extreme noise'
    if has_mixed_special(text):
        return 'Level 4 - Mixed special characters'
    if has_dot_dash(text):
        return 'Level 3 - Dot/Dash'
    if has_leet_heavy(text):
        return 'Level 2 - Leet nặng'
    if has_leet_light(text):
        return 'Level 1 - Leet nhẹ'

    return 'Level 0 - Formal'


# ──────────────────────────────────────────────
# Main pipeline
# ──────────────────────────────────────────────

def process_dataset(input_path: str, output_path: str):
    print(f"[INFO] Đọc file: {input_path}")
    df = pd.read_csv(input_path)

    # Kiểm tra các cột cần thiết
    required_cols = {'label', 'content'}
    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(f"[ERROR] File thiếu cột: {missing}")

    print(f"[INFO] Tổng số mẫu: {len(df)}")
    print(f"[INFO] Label 0: {(df['label'] == 0).sum()} | Label 1: {(df['label'] == 1).sum()}")

    # Phân loại
    def assign_level(row):
        if row['label'] == 0:
            return 'NONE'
        else:
            return classify_obfuscation(row['content'])

    df['obfuscation_level'] = df.apply(assign_level, axis=1)

    # Thống kê kết quả
    print("\n[RESULT] Phân phối obfuscation_level:")
    print(df['obfuscation_level'].value_counts().to_string())

    # Ghi output
    df.to_csv(output_path, index=False, encoding='utf-8-sig')
    print(f"\n[INFO] Đã lưu: {output_path}")


if __name__ == '__main__':
    input_csv = "temp.csv"      
    output_csv = "temp.csv" 
    
    process_dataset(input_csv, output_csv)

[INFO] Đọc file: temp.csv
[INFO] Tổng số mẫu: 10562
[INFO] Label 0: 5320 | Label 1: 5242

[RESULT] Phân phối obfuscation_level:
obfuscation_level
NONE                                  5320
Level 0 - Formal                      3221
Level 2 - Leet nặng                    782
Level 4 - Mixed special characters     724
Level 1 - Leet nhẹ                     273
Level 3 - Dot/Dash                     140
Level 5 - Extreme noise                102

[INFO] Đã lưu: temp.csv


3. Xóa các cột thừa

In [4]:
df=df.drop(columns=["source_dataset", "source_file","source_row_id"])
df.to_csv("D:\\UIT\\KLTN\\model\\base\\temp.csv", index=False)

Mô tả dataset

In [9]:
print("="*50)
print("1. THỐNG KÊ TỔNG QUAN")
print("="*50)
print(f"Tổng số mẫu: {len(df):,}")
print("\n--- Phân bố Nhãn (Label) ---")
print(df['label'].map({0: 'Ham (0)', 1: 'Spam/Smish (1)'}).value_counts().to_string())

print("\n--- Phân bố Mức độ Che giấu (Obfuscation Level) ---")
print(df['obfuscation_level'].value_counts().to_string())

print("\n--- Top 10 Danh mục (Category) ---")
print(df['category'].value_counts().head(10).to_string())

1. THỐNG KÊ TỔNG QUAN
Tổng số mẫu: 10,562

--- Phân bố Nhãn (Label) ---
label
Ham (0)           5320
Spam/Smish (1)    5242

--- Phân bố Mức độ Che giấu (Obfuscation Level) ---
obfuscation_level
NONE                                  5320
Level 0 - Formal                      3221
Level 2 - Leet nặng                    782
Level 4 - Mixed special characters     724
Level 1 - Leet nhẹ                     273
Level 3 - Dot/Dash                     140
Level 5 - Extreme noise                102

--- Top 10 Danh mục (Category) ---
category
Viễn thông             1256
Khác                    835
Tuyển dụng giả          709
Crypto / Đầu tư giả     636
Nội dung nhạy cảm       634
Cờ bạc / Betting        618
BHXH / Trợ cấp          589
Dịch vụ công giả        571
Giả mạo ngân hàng       528
P2P hard negative       501


In [10]:
print("="*50)
print("2. THỐNG KÊ ĐỘ DÀI TIN NHẮN")
print("="*50)

# Xử lý NaN (nếu có) và tính chiều dài chuỗi
df['length'] = df['content'].fillna('').apply(len)

# Thống kê chung
print("--- Thống kê chung ---")
print(f"Chiều dài trung bình: {df['length'].mean():.2f}")
print(f"Trung vị (Median):    {df['length'].median():.2f}")
print(f"Ngắn nhất (Min):      {df['length'].min()}")
print(f"Dài nhất (Max):       {df['length'].max()}")

# Thống kê theo nhãn
mean_spam_len = df[df['label'] == 1]['length'].mean()
mean_ham_len = df[df['label'] == 0]['length'].mean()

print("\n--- Chiều dài theo Nhãn ---")
print(f"Trung bình Spam (1): {mean_spam_len:.2f} ký tự")
print(f"Trung bình Ham (0):  {mean_ham_len:.2f} ký tự")

2. THỐNG KÊ ĐỘ DÀI TIN NHẮN
--- Thống kê chung ---
Chiều dài trung bình: 156.83
Trung vị (Median):    144.00
Ngắn nhất (Min):      2
Dài nhất (Max):       920

--- Chiều dài theo Nhãn ---
Trung bình Spam (1): 161.44 ký tự
Trung bình Ham (0):  152.28 ký tự


In [ ]:
"""
pmi_analysis.py
---------------
Phân tích PMI (Pointwise Mutual Information) cho dataset ViSmishDS.

3 tầng phân tích:
  1. PMI(token, label)      → từ nào đặc trưng nhất cho smishing vs ham
  2. PMI(token, obf_level)  → từ nào hay bị obfuscate ở từng level
  3. Combined score         → từ vừa đặc trưng smishing vừa hay bị che giấu
                              → đây là "từ nhạy cảm nguy hiểm nhất"

Công thức PPMI (Positive PMI — loại bỏ giá trị âm):
  PMI(t, c) = log2[ P(t,c) / (P(t) * P(c)) ]
  PPMI(t, c) = max(0, PMI(t, c))

Cách dùng:
  python pmi_analysis.py --input dataset.csv --output_dir ./pmi_results
  
  Bắt buộc: cột 'content', 'label', 'obfuscation_level' trong file CSV.
  Nếu chưa có cột obfuscation_level, chạy classify_obfuscation.py trước.
"""

# ──────────────────────────────────────────────
# Tiền xử lý & tokenize
# ──────────────────────────────────────────────

# Stopwords tiếng Việt cơ bản
STOPWORDS = {
    'là', 'và', 'của', 'có', 'cho', 'được', 'với', 'các', 'một', 'những',
    'này', 'đó', 'trong', 'đã', 'để', 'từ', 'theo', 'về', 'khi', 'hay',
    'bạn', 'tôi', 'anh', 'chị', 'em', 'mình', 'họ', 'chúng', 'ta',
    'không', 'chưa', 'cũng', 'đây', 'đó', 'vì', 'nếu', 'thì', 'mà',
    'rằng', 'như', 'lại', 'sẽ', 'đang', 'vẫn', 'rất', 'hơn', 'nhất',
    'nên', 'cần', 'phải', 'có thể', 'tuy nhiên', 'nhưng', 'hoặc',
}

MIN_FREQ = 5   # token xuất hiện ít hơn ngưỡng này sẽ bị loại
TOP_N    = 30  # số token top hiển thị trong mỗi bảng kết quả


def normalize_text(text: str) -> str:
    """Chuẩn hóa sơ bộ trước khi tokenize: lowercase, bỏ số đứng độc lập."""
    if not isinstance(text, str):
        return ''
    text = text.lower().strip()
    # Bỏ URL
    text = re.sub(r'http\S+|www\.\S+', ' ', text)
    # Bỏ số đứng độc lập (giá tiền, số điện thoại...) nhưng giữ leet bên trong từ
    text = re.sub(r'(?<!\w)\d+(?!\w)', ' ', text)
    # Bỏ ký tự đặc biệt thừa (giữ lại chữ cái, số trong từ, dấu tiếng Việt)
    text = re.sub(r'[^\w\s\u00c0-\u024f\u1ea0-\u1ef9]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def tokenize(text: str) -> list:
    """Tokenize tiếng Việt bằng underthesea, lọc stopwords và token quá ngắn."""
    text = normalize_text(text)
    if not text:
        return []
    tokens = word_tokenize(text, format='text').split()
    tokens = [t for t in tokens if t not in STOPWORDS and len(t) >= 2]
    return tokens


# ──────────────────────────────────────────────
# Tính PPMI
# ──────────────────────────────────────────────

def compute_ppmi(df: pd.DataFrame,
                 token_col: str,
                 context_col: str) -> pd.DataFrame:
    """
    Tính PPMI giữa token và context (label hoặc obf_level).
    
    Returns:
        DataFrame với cột: token, context, freq, ppmi
    """
    # Đếm co-occurrence P(t, c)
    pair_counts   = defaultdict(int)  # (token, context) → count
    token_counts  = defaultdict(int)  # token → count
    context_counts = defaultdict(int) # context → count
    total = 0

    for _, row in df.iterrows():
        tokens  = row[token_col]
        context = row[context_col]
        seen = set()
        for token in tokens:
            if token not in seen:            # mỗi token đếm 1 lần/doc (binary)
                pair_counts[(token, context)] += 1
                token_counts[token]           += 1
                context_counts[context]       += 1
                total += 1
                seen.add(token)

    # Lọc token hiếm
    valid_tokens = {t for t, c in token_counts.items() if c >= MIN_FREQ}

    # Tính PPMI
    records = []
    for (token, context), freq in pair_counts.items():
        if token not in valid_tokens:
            continue
        p_tc = freq / total
        p_t  = token_counts[token] / total
        p_c  = context_counts[context] / total
        pmi  = math.log2(p_tc / (p_t * p_c))
        ppmi = max(0.0, pmi)
        records.append({
            'token':   token,
            'context': context,
            'freq':    freq,
            'ppmi':    round(ppmi, 4)
        })

    return pd.DataFrame(records)


# ──────────────────────────────────────────────
# Main analysis
# ──────────────────────────────────────────────

def run_analysis(input_path: str, output_dir: str):
    os.makedirs(output_dir, exist_ok=True)

    print(f"[INFO] Đọc file: {input_path}")
    df = pd.read_csv(input_path)

    required = {'content', 'label', 'obfuscation_level'}
    missing  = required - set(df.columns)
    if missing:
        raise ValueError(f"[ERROR] File thiếu cột: {missing}")

    print(f"[INFO] Tổng số mẫu: {len(df)}")

    # ── Tokenize ──
    print("[INFO] Đang tokenize... (có thể mất vài phút)")
    df['tokens'] = df['content'].apply(tokenize)

    # ── Tầng 1: PMI(token, label) ──────────────────────────────────────────
    print("[INFO] Tính PMI(token ↔ label)...")
    pmi_label = compute_ppmi(df, 'tokens', 'label')

    # Top token đặc trưng cho smishing (label=1)
    smishing_words = (
        pmi_label[pmi_label['context'] == 1]
        .sort_values('ppmi', ascending=False)
        .head(TOP_N)
        .reset_index(drop=True)
    )
    smishing_words.index += 1
    smishing_words.columns = ['Từ', 'Label', 'Tần suất', 'PPMI']

    # Top token đặc trưng cho ham (label=0)
    ham_words = (
        pmi_label[pmi_label['context'] == 0]
        .sort_values('ppmi', ascending=False)
        .head(TOP_N)
        .reset_index(drop=True)
    )
    ham_words.index += 1
    ham_words.columns = ['Từ', 'Label', 'Tần suất', 'PPMI']

    # ── Tầng 2: PMI(token, obf_level) ──────────────────────────────────────
    print("[INFO] Tính PMI(token ↔ obfuscation_level)...")
    df_smish = df[df['label'] == 1].copy()
    pmi_obf  = compute_ppmi(df_smish, 'tokens', 'obfuscation_level')

    obf_top = {}
    level_order = ['FORMAL', 'LEET_LIGHT', 'LEET_HEAVY', 'DOT_DASH', 'MIXED_SPECIAL', 'EXTREME_NOISE']
    for level in level_order:
        top = (
            pmi_obf[pmi_obf['context'] == level]
            .sort_values('ppmi', ascending=False)
            .head(TOP_N)
            .reset_index(drop=True)
        )
        top.index += 1
        top.columns = ['Từ', 'Obf Level', 'Tần suất', 'PPMI']
        obf_top[level] = top

    # ── Tầng 3: Combined score ──────────────────────────────────────────────
    print("[INFO] Tính Combined score...")

    # PPMI smishing của từng token
    smishing_ppmi = (
        pmi_label[pmi_label['context'] == 1]
        [['token', 'ppmi', 'freq']]
        .rename(columns={'ppmi': 'ppmi_smishing', 'freq': 'freq_smishing'})
    )

    # PPMI obf tổng hợp: lấy max PPMI qua tất cả level cho mỗi token
    obf_max = (
        pmi_obf.groupby('token')['ppmi']
        .max()
        .reset_index()
        .rename(columns={'ppmi': 'ppmi_obf_max'})
    )
    # Level nào cho max PPMI
    obf_best_level = (
        pmi_obf.loc[pmi_obf.groupby('token')['ppmi'].idxmax()]
        [['token', 'context']]
        .rename(columns={'context': 'best_obf_level'})
    )

    combined = (
        smishing_ppmi
        .merge(obf_max,        on='token', how='inner')
        .merge(obf_best_level, on='token', how='left')
    )
    combined['combined_score'] = (
        combined['ppmi_smishing'] * combined['ppmi_obf_max']
    ).round(4)
    combined = (
        combined
        .sort_values('combined_score', ascending=False)
        .head(TOP_N)
        .reset_index(drop=True)
    )
    combined.index += 1
    combined.columns = ['Từ', 'PPMI Smishing', 'Tần suất', 'PPMI Obf (max)', 'Level Obf Cao Nhất', 'Combined Score']

    # ── Ghi kết quả ra CSV ──────────────────────────────────────────────────
    path_smish  = os.path.join(output_dir, 'pmi_smishing_words.csv')
    path_ham    = os.path.join(output_dir, 'pmi_ham_words.csv')
    path_obf    = os.path.join(output_dir, 'pmi_obf_level_words.csv')
    path_combo  = os.path.join(output_dir, 'pmi_combined_dangerous_words.csv')

    smishing_words.to_csv(path_smish, encoding='utf-8-sig')
    ham_words.to_csv(path_ham,        encoding='utf-8-sig')

    # Gộp tất cả obf level vào 1 file
    all_obf = pd.concat(obf_top.values(), ignore_index=False)
    all_obf.to_csv(path_obf, encoding='utf-8-sig')

    combined.to_csv(path_combo, encoding='utf-8-sig')

    # ── In tóm tắt ra terminal ──────────────────────────────────────────────
    print("\n" + "="*60)
    print(f"  TOP {TOP_N} TỪ ĐẶC TRƯNG CHO SMISHING (label=1)")
    print("="*60)
    print(smishing_words[['Từ', 'Tần suất', 'PPMI']].to_string())

    print("\n" + "="*60)
    print(f"  TOP {TOP_N} TỪ NGUY HIỂM NHẤT (smishing + bị obfuscate)")
    print("="*60)
    print(combined[['Từ', 'PPMI Smishing', 'PPMI Obf (max)', 'Level Obf Cao Nhất', 'Combined Score']].to_string())

    print("\n" + "="*60)
    print("  TOP TỪ BỊ OBFUSCATE THEO TỪNG LEVEL")
    print("="*60)
    for level in level_order:
        if not obf_top[level].empty:
            top5 = obf_top[level].head(5)['Từ'].tolist()
            print(f"  {level:<16}: {', '.join(top5)}")

    print(f"\n[INFO] Đã lưu kết quả vào thư mục: {output_dir}/")
    print(f"       ├── pmi_smishing_words.csv")
    print(f"       ├── pmi_ham_words.csv")
    print(f"       ├── pmi_obf_level_words.csv")
    print(f"       └── pmi_combined_dangerous_words.csv  ← quan trọng nhất")


if __name__ == '__main__':
    parser = argparse.ArgumentParser(description='PMI Analysis cho dataset ViSmishDS')
    parser.add_argument('--input',      required=True, help='File CSV đầu vào (đã có obfuscation_level)')
    parser.add_argument('--output_dir', default='./pmi_results', help='Thư mục lưu kết quả (mặc định: ./pmi_results)')
    args = parser.parse_args()

    run_analysis(args.input, args.output_dir)

3. TÍNH TOÁN CHỈ SỐ PMI (TỪ KHÓA LỪA ĐẢO)
   Từ khóa (Feature Word)  Chỉ số PMI
1               987654321    1.010694
2                    pc03    1.010694
3                   asdfg    1.010694
4                      88    1.010694
5                   hotro    1.010694
6                 techcom    1.010694
7                      eu    1.010694
8              1234567890    1.010694
9                    asia    1.010694
10                  trịnh    1.010694
11            84987654321    1.010694
12                   t1en    1.010694
13                   giấc    1.010694
14                 update    1.010694
15                   bank    1.010694
